# CNN for CIFAR10

In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#Immage => Scaling(Min-Max Scaling (0,1) => Normalize => (-1,1)
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    
])

# We need not to use train_test_split and torch's Datset module to create training sets since CIFAR10 already has inside it
trainset=CIFAR10(root="./data",train=True, download=True, transform=transform)
testset=CIFAR10(root="./data",train=False, download=True, transform=transform)


In [4]:
trainLoader=DataLoader(trainset, batch_size=64,shuffle=True)
testLoader=DataLoader(testset, batch_size=64)

In [18]:
#Defining our Model

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            #1st Layer
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #Kernel size=2 and Stride=2

            #2nd Layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), #Kernel size=2 and Stride=2

            #3rd Layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) #Kernel size=2 and Stride=2
        )

        self.fc_layers=nn.Sequential(

            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
            
            
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)  #flattening
        x=self.fc_layers(x)

        return x

In [19]:
#Building our CNN Model

model=CNN()

# loss, Optimizer
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [26]:
#Training our Model

# Training Our CNN Model, we can also apply validation here(HomeWork)

epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0 #Total Training Loss 1 Epoch
    model.train()
    
    for images,labels in trainLoader:
        #images= Features of 1 batch, labels= Outputs/Labels of 1 batch
        optimizer.zero_grad()
        
        output=model(images)  # Forward Propagation  or model.forward(images) works too but not recommendeded
        loss=criterion(output, labels)  #Loss Computation
        loss.backward() #Back Propagation
        optimizer.step() #Updating the parameters

        epoch_training_loss+=loss.item() #convert from tensor to python float

    print(f"Epoch={epoch+1}/{epochs} ==> Training Loss: {epoch_training_loss/len(trainLoader)}")

Epoch=1/10 ==> Training Loss: 0.08119987299048896
Epoch=2/10 ==> Training Loss: 0.0833227969161199
Epoch=3/10 ==> Training Loss: 0.07780827542759783
Epoch=4/10 ==> Training Loss: 0.07413541372743962
Epoch=5/10 ==> Training Loss: 0.07049022034070243
Epoch=6/10 ==> Training Loss: 0.07206042450147054
Epoch=7/10 ==> Training Loss: 0.06144252649332454
Epoch=8/10 ==> Training Loss: 0.06486027142973236
Epoch=9/10 ==> Training Loss: 0.0643906749752493
Epoch=10/10 ==> Training Loss: 0.0653398853922894


In [27]:
# Evaluating or Testing our CNN Model

model.eval()

correct_labels=0
total_labels=0

with torch.no_grad():

    for images,labels in testLoader:
        outputs=model(images) # model.forward(images) works too but not recommendeded
        max_val, predicted_class = torch.max(outputs,1)

        correct_labels+=(predicted_class==labels).sum().item() #for one batch
        total_labels+=labels.size(0) #actual number of samples in each batch

print("Total Values:", total_labels)
print("Correct Values:", correct_labels)
print("Accuracy Score:",(correct_labels/total_labels)*100)

Total Values: 10000
Correct Values: 7389
Accuracy Score: 73.89
